# Optimization analysis

This notebook runs the project's reproducible clean-room analysis pipeline and explains the resulting network decisions. Model construction, solving, independent validation, enumeration, sensitivity analysis, and visualization remain in project modules rather than notebook cells.

## Provenance and claims boundary

The historical notebook is reference context only and is not executed or reproduced here. This analysis uses only the independently created scenario files in data/scenario.

All entities and strategic financial inputs are scenario assumptions; lane distances and unit costs are deterministic derivatives of schematic synthetic coordinates; and reported differences from the baseline are scenario cost reductions, not verified business savings. The deterministic result does not represent a deployed company network.

The notebook contains no saved execution history or output values. Run the cells to regenerate current results from source data.

## Setup and scenario load

Run from the repository root. The first cell explicitly loads the same validated base scenario used by the command-line workflow, so subsequent interpretation remains traceable to the input tables.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
try:
    from IPython.display import Image, Markdown, display
except ImportError:
    def display(value):
        print(value)

    def Markdown(text):
        return text

    def Image(*, filename):
        return f"[image: {filename}]"

start_path = Path.cwd().resolve()
candidate_roots = (start_path, start_path.parent)
DEV_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "src").is_dir() and (path / "data" / "scenario").is_dir()
    ),
    None,
)
if DEV_ROOT is None:
    raise RuntimeError(
        "Start Jupyter from the repository root or its notebooks directory."
    )
if str(DEV_ROOT) not in sys.path:
    sys.path.insert(0, str(DEV_ROOT))

from run_analysis import run_analysis
from src.data_loader import apply_scenario, load_network_data, load_scenarios

DATA_DIR = DEV_ROOT / "data" / "scenario"
OUTPUT_DIR = DEV_ROOT / ".private_outputs" / "module_7_25C"

base_inputs = load_network_data(DATA_DIR)
scenario_definitions = load_scenarios(DATA_DIR / "scenarios.csv")
base_definition = next(
    item for item in scenario_definitions if item.scenario_id == "DEMAND_BASE"
)
base_data = apply_scenario(base_inputs, base_definition)

## Reproduce the complete analysis

The orchestration function loads and validates inputs, builds and solves the Pyomo MILP, refuses non-optimal solver status, recomputes every solution check, evaluates the all-facilities-open baseline, exhaustively enumerates binary facility subsets with independent transportation LPs, runs declared sensitivities, verifies the tiny known case, and writes tables and figures.

In [ ]:
summary = run_analysis(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    solver_name="appsi_highs",
)

solver_summary = summary["solver"]
run_summary = pd.Series(
    {
        "build_status": summary["build_status"],
        "public_framing": summary["public_framing"],
        "scenario": summary["scenario_name"],
        "solver_interface": solver_summary["solver_name"],
        "solver_version": solver_summary["solver_version"],
        "solver_status": solver_summary["solver_status"],
        "termination_condition": solver_summary["termination_condition"],
        "relative_mip_gap": solver_summary["relative_mip_gap"],
        "objective_scenario_usd_per_year": solver_summary["objective_value"],
        "selected_facilities": ", ".join(summary["selected_facilities"]),
    },
    name="value",
).to_frame()
display(run_summary)

## Decision tables

The facility table links each binary opening decision to capacity, assigned flow, utilization, and fixed-cost contribution. The shipment table contains only active arcs. Customer differences should remain within the documented numerical tolerance.

In [ ]:
TABLES_DIR = OUTPUT_DIR / "tables"
facility_decisions = pd.read_csv(TABLES_DIR / "facility_decisions.csv")
active_shipments = pd.read_csv(TABLES_DIR / "active_shipments.csv")
customer_service = pd.read_csv(TABLES_DIR / "customer_service.csv")
cost_breakdown = pd.read_csv(TABLES_DIR / "cost_breakdown.csv")
baseline_summary = pd.read_csv(TABLES_DIR / "baseline_summary.csv")
sensitivity_results = pd.read_csv(TABLES_DIR / "sensitivity_results.csv")

display(
    facility_decisions[
        [
            "facility_id",
            "open",
            "capacity_units_per_year",
            "assigned_flow_units_per_year",
            "utilization",
            "unused_capacity_units_per_year",
            "fixed_cost_contribution_usd_per_year",
            "capacity_binding",
        ]
    ].round(4)
)

display(
    active_shipments[
        [
            "origin",
            "destination",
            "shipment_units_per_year",
            "unit_cost_usd_per_unit",
            "transport_cost_usd_per_year",
        ]
    ]
    .sort_values("shipment_units_per_year", ascending=False)
    .reset_index(drop=True)
    .round(2)
)

display(
    customer_service[
        [
            "customer_id",
            "demand_units_per_year",
            "delivered_units_per_year",
            "difference_units_per_year",
        ]
    ].round(6)
)

## Cost decomposition and baseline

The objective is the sum of annual fixed facility cost and annual shipment cost. The transparent baseline fixes every candidate facility open and then minimizes shipment cost. Its difference from the optimized design is meaningful only inside this synthetic scenario.

In [ ]:
display(cost_breakdown.round(2))

display(
    baseline_summary[
        [
            "baseline_id",
            "definition",
            "facility_count",
            "fixed_cost_usd_per_year",
            "transport_cost_usd_per_year",
            "objective_usd_per_year",
            "optimized_objective_usd_per_year",
            "scenario_cost_difference_usd_per_year",
            "scenario_cost_reduction_relative_to_baseline",
        ]
    ].round(4)
)

## Independent quality gates

Solver output is not accepted on its own. The project recomputes demand balance, capacity use, closed-facility flow, nonnegativity, binary domains, and both objective components from decision values. Exhaustive facility-subset enumeration uses independently formulated SciPy transportation LPs; it is practical here because six binary decisions produce only 64 subsets, not as a method for large networks. Module 7.5C also reproduced the results with a solver-engine-free minimum-cost-flow path. The tiny two-facility regression case has a manually derived expected optimum.

In [ ]:
def read_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

validation = read_json(OUTPUT_DIR / "validation_report.json")
enumeration = read_json(OUTPUT_DIR / "enumeration_validation.json")
tiny_case = read_json(OUTPUT_DIR / "tiny_known_case_validation.json")
infeasibility_demo = read_json(OUTPUT_DIR / "infeasibility_demo.json")

quality_gates = pd.DataFrame(
    [
        {
            "gate": "solver-certified optimum",
            "passed": solver_summary["termination_condition"] == "optimal",
            "detail": solver_summary["termination_condition"],
        },
        {
            "gate": "independent feasibility and objective check",
            "passed": validation["passed"],
            "detail": (
                "objective absolute difference = "
                f"{validation['objective_absolute_difference']:.6g}"
            ),
        },
        {
            "gate": "closed facilities ship zero",
            "passed": (
                validation["max_closed_facility_flow"]
                <= validation["feasibility_tolerance"]
            ),
            "detail": (
                "maximum closed-facility flow = "
                f"{validation['max_closed_facility_flow']:.6g}"
            ),
        },
        {
            "gate": "exhaustive subset enumeration agreement",
            "passed": enumeration["passed"],
            "detail": (
                f"{enumeration['feasible_subsets']} feasible of "
                f"{enumeration['total_subsets']} subsets"
            ),
        },
        {
            "gate": "tiny known-case regression",
            "passed": tiny_case["passed"],
            "detail": (
                "solver and enumeration reproduce the manual fixture optimum"
            ),
        },
        {
            "gate": "controlled infeasibility precheck",
            "passed": infeasibility_demo["passed"],
            "detail": infeasibility_demo["observed_status"],
        },
    ]
).set_index("gate")
display(quality_gates)

## Decision-relevant sensitivity

The registered scenarios vary demand, transport cost, fixed cost, or capacity one dimension at a time. The important structural signal is whether the selected facility configuration changes, not merely whether total scenario cost rises or falls.

In [ ]:
display(
    sensitivity_results[
        [
            "scenario_id",
            "demand_multiplier",
            "capacity_multiplier",
            "fixed_cost_multiplier",
            "transport_cost_multiplier",
            "objective_usd_per_year",
            "open_facilities",
            "binding_facilities",
            "network_utilization",
            "configuration_changed_from_base",
        ]
    ].round(4)
)

## Facility tradeoff view

Selection reflects the joint effect of fixed cost, available capacity, and customer-specific shipment economics. The table below pairs each decision with its scenario inputs and descriptive arc-cost statistics. These summaries support interpretation but do not replace the optimization model or prove that any single factor caused a decision.

In [ ]:
arc_cost_statistics = (
    base_data.transport_costs.groupby("facility_id")["unit_cost_usd_per_unit"]
    .agg(
        minimum_unit_cost_usd_per_unit="min",
        mean_unit_cost_usd_per_unit="mean",
        maximum_unit_cost_usd_per_unit="max",
    )
    .reset_index()
)

facility_tradeoffs = (
    facility_decisions[
        [
            "facility_id",
            "open",
            "assigned_flow_units_per_year",
            "utilization",
            "capacity_binding",
        ]
    ]
    .merge(
        base_data.facilities[
            [
                "facility_id",
                "capacity_units_per_year",
                "fixed_cost_usd_per_year",
            ]
        ],
        on="facility_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        arc_cost_statistics,
        on="facility_id",
        how="left",
        validate="one_to_one",
    )
)
display(facility_tradeoffs.round(3))

## Generated figures

The production visualization module creates a schematic optimized network, cost comparison, facility-utilization chart, and sensitivity comparison. The network figure is not a real map.

In [ ]:
figure_specs = [
    ("Optimized schematic network", "optimized_network.png"),
    ("Optimized and baseline cost breakdown", "cost_breakdown.png"),
    ("Facility utilization", "facility_utilization.png"),
    ("Sensitivity comparison", "sensitivity_comparison.png"),
]

for title, filename in figure_specs:
    figure_path = OUTPUT_DIR / "figures" / filename
    if not figure_path.is_file():
        raise FileNotFoundError(f"Expected generated figure is missing: {figure_path}")
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_path)))

## Interpretation limits

This is a deterministic, single-period, single-echelon demonstration with synthetic demand, capacity, coordinates, and cost assumptions. It omits real road routing, freight contracts, inventory dynamics, service uncertainty, and production deployment. Robust or stochastic optimization is a possible future extension, but the present claims are limited to solver-certified and independently verified results for the declared scenarios.